# Train one agro-ai crop expert on Kaggle

Before running:

1. **Settings → Accelerator:** GPU (T4 or P100). **Internet:** on if you use `REPO_URL` (the toggle only appears after phone verification).
2. **Add Input** the datasets the crop needs. They are used in place from `/kaggle/input`, nothing is downloaded:

| Crop | Inputs |
|---|---|
| cassava | competition `cassava-leaf-disease-classification` (accept its rules first) |
| maize | `smaranjitghose/corn-or-maize-leaf-disease-dataset`, `vipoooool/new-plant-diseases-dataset` |
| tomato, potato, apple | `vipoooool/new-plant-diseases-dataset` (optionally `nirmalsankalana/plant-diseases-training-dataset`, `salmasyed1360/plant-diseases-100k-labelled-images`) |
| tea | `shashwatwork/identifying-disease-in-tea-leafs` |
| coffee | `nirmalsankalana/rocole-a-robusta-coffee-leaf-images-dataset` |
| cashew | `olaidegabriel/cashew-disease-detection-dataset` |

3. Make the code available: either set `REPO_URL` below to the GitHub repo (needs Internet on), or upload the repo folder as a private Kaggle dataset and add it as an input (works with Internet off; it is found anywhere under `/kaggle/input`).
4. **With Internet off**, the MobileNetV3 ImageNet weights can't be downloaded. Put `weights_mobilenet_v3_large_224_1.0_float_no_top_v2.h5` (from https://storage.googleapis.com/tensorflow/keras-applications/mobilenet_v3/weights_mobilenet_v3_large_224_1.0_float_no_top_v2.h5) in the repo's `weights/` folder before zipping.

Outputs land in `/kaggle/working/models/` (download them from the notebook's Output tab).

In [ ]:
CROP = "cassava"
REPO_URL = ""                              # e.g. "https://github.com/<you>/agro-ai.git" (needs Internet on)
# The uploaded code is found anywhere under /kaggle/input; this only picks between several copies.
CODE_INPUT = "/kaggle/input/datasets/alainregis/agroai-src"
EXTRA_TRAIN_ARGS = "--mixed-precision"      # e.g. "--coord-attention --epochs-finetune 20"

In [ ]:
import glob, os, shutil, socket, subprocess, sys, zipfile

def internet_on():
    try:
        socket.create_connection(("github.com", 443), timeout=5).close()
        return True
    except OSError:
        return False

def _repo_dirs(base, max_depth=5):
    """Folders under base holding both dataset_factory.py and agro_ai/, shallowest first."""
    for depth in range(0, max_depth + 1):
        pattern = os.path.join(base, *(["*"] * depth), "dataset_factory.py")
        hits = [os.path.dirname(p) for p in glob.glob(pattern)
                if os.path.isdir(os.path.join(os.path.dirname(p), "agro_ai"))]
        if hits:
            return hits
    return []

def find_uploaded_code():
    """Find the repo inside /kaggle/input wherever Kaggle mounted it (any dataset name,
    /kaggle/input/datasets/<owner>/<name>, an extra agro-ai/ folder from the zip,
    or a zip Kaggle left unextracted)."""
    hits = _repo_dirs("/kaggle/input")
    if hits:
        preferred = [h for h in hits if h.startswith(CODE_INPUT)]
        return (preferred or hits)[0]
    for depth in range(1, 5):
        for archive in glob.glob("/kaggle/input/" + "*/" * depth + "*.zip"):
            with zipfile.ZipFile(archive) as zf:
                if any(n.endswith("dataset_factory.py") for n in zf.namelist()):
                    out = "/kaggle/working/agro-ai-unzipped"
                    zf.extractall(out)
                    hits = _repo_dirs(out)
                    if hits:
                        return hits[0]
    return None

def show_inputs(max_depth=3):
    for root, dirs, files in os.walk("/kaggle/input"):
        depth = root[len("/kaggle/input"):].count(os.sep)
        print("  " * depth + os.path.basename(root) + "/", f"({len(files)} files)", files[:5])
        if depth >= max_depth:
            dirs[:] = []
        else:
            dirs.sort()
            del dirs[8:]

CODE = "/kaggle/working/agro-ai"
if not os.path.isdir(CODE):
    found = find_uploaded_code()
    if found:
        print("using uploaded code from", found)
        shutil.copytree(found, CODE)
    elif REPO_URL and internet_on():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, CODE], check=True)
    else:
        print("What is attached under /kaggle/input:")
        show_inputs()
        raise SystemExit(
            "Can't find the code: no folder under /kaggle/input contains both dataset_factory.py and agro_ai/. "
            "Add your agro-ai-code dataset with 'Add Input' (check the tree printed above), "
            "or turn on Settings -> Internet and set REPO_URL.")
# No pip install: running from the repo folder is enough, and it also works with Internet off.
os.chdir(CODE)
sys.path.insert(0, CODE)
!python dataset_factory.py --list-sources

In [ ]:
# Only sources attached as inputs are used; the others are skipped with a note.
!python dataset_factory.py --crops {CROP} --no-download --workers 4
if _exit_code:
    raise SystemExit("dataset_factory failed, see the error above")

In [ ]:
import json, pathlib

DATA = pathlib.Path("/kaggle/working/agro-data/processed") / CROP
report = json.loads((DATA / "report.json").read_text())
print("images:", report["total"], "| duplicates removed:", report["duplicates_removed"], "| rejected:", report["rejected"])
for label, c in sorted(report["classes"].items()):
    print(f"{label:<40} train={c.get('train', 0):>6} val={c.get('val', 0):>5} test={c.get('test', 0):>5} "
          f"field={c.get('domain_field', 0):>6} lab={c.get('domain_lab', 0):>6}")
print("\n".join(report["warnings"]) or "no warnings")

In [ ]:
!python -m agro_ai.train.train --data {DATA} {EXTRA_TRAIN_ARGS}
if _exit_code:
    raise SystemExit("training failed, see the error above")

In [ ]:
manifest = json.loads(pathlib.Path("/kaggle/working/models/manifest.json").read_text())
latest = [m for m in manifest["models"] if m["name"] == CROP][-1]
print(json.dumps({k: latest[k] for k in ("name", "version", "recommended", "metrics")}, indent=2))
for f in latest["files"]:
    print(f"{f['variant']:<5} {f['bytes'] / 1e6:6.2f} MB  {f['path']}")

In [ ]:
# Keep the output small: drop the processed image copies, keep only models/.
shutil.rmtree("/kaggle/working/agro-data", ignore_errors=True)
shutil.rmtree(CODE, ignore_errors=True)